In [1]:
using MarineHydro
using PyCall
using DimensionalData
cpt = pyimport("capytaine")

# Description of problem
h = Inf # sea depth [m]
omegas = [1.03, 1.7] # frequencies [rad/s]
betas = [0.0, pi/6] # incident wave angle [rad]
beta = betas[1]
t_DOFs = ["Heave"] # translational DOFs
r_DOFs = ["Pitch"] # rotational DOFs
DOFs = [t_DOFs; r_DOFs] # all DOFs

# Create Capytaine Mesh object
radius = 1.5  
center = (0.0,0.0,0.0) 
rotation_center = (1.0, 1.0, 0.0) # off set for nonzero off-diagoinal elements
len = 2.5
faces_max_radius = 0.5
cptmesh = cpt.meshes.predefined.mesh_horizontal_cylinder(
            radius=radius,
            center=center, 
            length=len, 
            faces_max_radius = faces_max_radius
            ).keep_immersed_part(inplace=true)



PyObject Mesh(vertices=[[... 83 vertices ...]], faces=[[... 80 faces ...]], name="cylinder_0")

In [2]:
# Solve using Capytaine

# Create FloatingBody object
cptbody = cpt.FloatingBody(mesh=cptmesh)
cptbody.center_of_mass = center
cptbody.rotation_center = rotation_center
foreach(dof -> cptbody.add_translation_dof(name=dof), t_DOFs)
foreach(dof -> cptbody.add_rotation_dof(name=dof), r_DOFs)
cptbody.active_dofs = DOFs
cptbody.name = "Horizontal Cylinder 1"

# Setup and solve BEM problems
solver = cpt.BEMSolver()
dof_list = cptbody.active_dofs
xr = pyimport("xarray")
test_matrix = xr.Dataset(coords=Dict("omega" => omegas, "wave_direction" => betas, "radiating_dof" => DOFs))
cptresults = cpt.BEMSolver().fill_dataset(test_matrix, cptbody, method="direct")

# Get Capytaine values
# A_cpt = cptresults.added_mass
# B_cpt = cptresults.radiation_damping
# F_FK_cpt = cptresults.Froude_Krylov_force 
# F_D_cpt = cptresults.diffraction_force
# F_ex_cpt = cptresults.excitation_force

Solving BEM problems ---------------------------------------- 100% 0:00:00


PyObject <xarray.Dataset> Size: 740B
Dimensions:              (omega: 2, radiating_dof: 2, influenced_dof: 2,
                          wave_direction: 2)
Coordinates:
    g                    float64 8B 9.81
    rho                  float64 8B 1e+03
    body_name            <U21 84B 'Horizontal Cylinder 1'
    water_depth          float64 8B inf
    forward_speed        float64 8B 0.0
  * wave_direction       (wave_direction) float64 16B 0.0 0.5236
  * omega                (omega) float64 16B 1.03 1.7
  * radiating_dof        (radiating_dof) object 16B 'Heave' 'Pitch'
  * influenced_dof       (influenced_dof) object 16B 'Heave' 'Pitch'
    period               (omega) float64 16B 6.1 3.696
    wavenumber           (omega) float64 16B 0.1081 0.2946
    wavelength           (omega) float64 16B 58.1 21.33
Data variables:
    added_mass           (omega, radiating_dof, influenced_dof) float64 64B 6...
    radiation_damping    (omega, radiating_dof, influenced_dof) float64 64B 1...
    diffraction_force    (omega, wave_direction, influenced_dof) complex128 128B ...
    Froude_Krylov_force  (omega, wave_direction, influenced_dof) complex128 128B ...
    excitation_force     (omega, wave_direction, influenced_dof) complex128 128B ...
Attributes: (12/16)
    start_of_computation:                     2026-03-20T13:21:35.735635
    green_function:                           Delhommeau
    tabulation_nr:                            676
    tabulation_rmax:                          100.0
    tabulation_nz:                            372
    tabulation_zmin:                          -251.0
    ...                                       ...
    gf_singularities:                         low_freq
    engine:                                   BasicMatrixEngine
    matrix_cache_size:                        1
    linear_solver:                            lu_decomposition
    creation_of_dataset:                      2026-03-20T13:21:35.784544
    capytaine_version:                        2.2.1

In [3]:
# Solve using MarineHydro

# Create Mesh struct
mhmesh = Mesh(cptmesh)

# Create FloatingBody struct
floatingbody1 = FloatingBody(mhmesh,
    Symbol.(DOFs),
    [rotation_center...],
    "Horizontal Cylinder 1")

# Create parameters NamedTuple
parameters = (wave_frequencies=omegas, 
wave_directions=betas,
radiating_dofs=Symbol.(DOFs))

# Create DimStack of results
mhresults = compute_hydrodynamic_coefficients(parameters, floatingbody1)


┌ 2×2×2×2 DimStack ┐
├──────────────────┴───────────────────────────────────────────────────── dims ┐
  ↓ influenced_dofs Categorical{Symbol} [:Heave, :Pitch] ForwardOrdered,
  → radiating_dofs Categorical{Symbol} [:Heave, :Pitch] ForwardOrdered,
  ↗ wave_frequencies Sampled{Float64} [1.03, 1.7] ForwardOrdered Irregular Points,
  ⬔ wave_directions Sampled{Float64} [0.0, 0.5235987755982988] ForwardOrdered Irregular Points
├────────────────────────────────────────────────────────────────────── layers ┤
  :added_mass          eltype: Float64 dims: influenced_dofs, radiating_dofs, wave_frequencies size: 2×2×2
  :radiation_damping   eltype: Float64 dims: influenced_dofs, radiating_dofs, wave_frequencies size: 2×2×2
  :excitation_force    eltype: ComplexF64 dims: influenced_dofs, wave_frequencies, wave_directions size: 2×2×2
  :diffraction_force   eltype: ComplexF64 dims: influenced_dofs, wave_frequencies, wave_directions size: 2×2×2
  :Froude_Krylov_force eltype: ComplexF64 dims: influenced

In [4]:
# Compare added_mass
omega_num = 2

display("Capytaine Added Mass")
display(cptresults.added_mass.sel(omega=omegas[omega_num]).values)

display("MarineHydro Added Mass")
display(collect(mhresults.added_mass[wave_frequencies = At(omegas[omega_num])]))

"Capytaine Added Mass"

2×2 Matrix{Float64}:
 5305.91  5305.91
 5305.91  8670.69

"MarineHydro Added Mass"

2×2 Matrix{Float64}:
 5313.38  5313.38
 5313.38  8676.46

In [5]:
# Multiple Body Case

# Create Capytaine Mesh object
radius = 1.5  
sep_dis = 50.0
center = (sep_dis,sep_dis,0.0) 
rotation_center = (sep_dis + 1.0, sep_dis + 1.0, 0.0) # off set for nonzero off-diagoinal elements
len = 2.5
faces_max_radius = 0.5
cptmesh2 = cpt.meshes.predefined.mesh_horizontal_cylinder(
            radius=radius,
            center=center, 
            length=len, 
            faces_max_radius = faces_max_radius
            ).keep_immersed_part(inplace=true)


# Create FloatingBody object
cptbody2 = cpt.FloatingBody(mesh=cptmesh2)
cptbody2.center_of_mass = center
cptbody2.rotation_center = rotation_center
foreach(dof -> cptbody2.add_translation_dof(name=dof), t_DOFs)
foreach(dof -> cptbody2.add_rotation_dof(name=dof), r_DOFs)
cptbody2.active_dofs = DOFs
cptbody2.name = "Horizontal Cylinder 2"

# Create Mesh struct
mhmesh2 = Mesh(cptmesh2)

# Create FloatingBody struct
floatingbody2 = FloatingBody(mhmesh2,
    Symbol.(DOFs),
    [rotation_center...],
    "Horizontal Cylinder 2")

floatingbodylist = [floatingbody1, floatingbody2]


2-element Vector{FloatingBody}:
 FloatingBody(Mesh([-1.25 -1.4623918682727355 -0.33378140093447134; -1.25 -1.4623918682727357 0.0; … ; 1.25 1.4623918682727355 -0.3337814009344717; 1.25 1.4623918682727355 -2.7755575615628914e-17], [21 33 32 12; 33 42 41 32; … ; 82 79 78 81; 59 56 55 58], [-0.9375 0.3254128043381684 -1.4257266509268143; -0.3125 0.3254128043381684 -1.4257266509268143; … ; 1.25 1.2349086887636433 -0.14092992483899916; 1.25 -1.2349086887636436 -0.14092992483899902], [0.0 0.2225209339563142 -0.9749279121818238; 0.0 0.2225209339563142 -0.9749279121818238; … ; 1.0 0.0 0.0; 1.0 0.0 0.0], [0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.41722675116808955, 0.41722675116808955  …  0.054235467389694765, 0.02711773369484739, 0.02711773369484736, 0.05423546738969476, 0.05423546738969476, 0.05423546738969479, 0.08135320108454216, 0.08135320108454208, 0.1355886684742369

In [6]:
function combine_meshes(meshlist::Vector{Mesh})

    # Make lists
    vetrices_list = [mesh.vertices for mesh in meshlist]
    faces_list = [mesh.faces for mesh in meshlist]
    centers_list = [mesh.centers for mesh in meshlist]
    normals_list = [mesh.normals for mesh in meshlist]
    areas_list = [mesh.areas for mesh in meshlist]
    radii_list = [mesh.radii for mesh in meshlist]
    nvetrices_list = [mesh.nvertices for mesh in meshlist]
    nfaces_list = [mesh.nfaces for mesh in meshlist]

    # Concatinate
    new_vertices = reduce(vcat, vetrices_list)
    new_centers = reduce(vcat, centers_list)
    new_normals = reduce(vcat, normals_list)
    new_areas = reduce(vcat, areas_list)
    new_radii = reduce(vcat, radii_list)

    # Sum 
    new_nvetrices = sum(nvetrices_list) 
    new_nfaces = sum(nfaces_list) 

    # Recount faces
    cum_nfaces_list = cumsum(nfaces_list)
    new_faces = zeros(Int, new_nfaces,4)
    for (nfaces_index,nfaces) in enumerate(nfaces_list)
        if nfaces_index == 1
            nbf = 0
            nbv = 0
        else
            nbf = cum_nfaces_list[nfaces_index-1]
            nbv = nvetrices_list[nfaces_index-1]
        end
        new_faces[nbf+1:nbf+nfaces,:] = faces_list[nfaces_index] .+ nbv
    end

    # Define combined Mesh struct
    return Mesh(new_vertices,
        new_faces,
        new_centers,
        new_normals,
        new_areas,
        new_radii,
        new_nvetrices,
        new_nfaces)    
end



combine_meshes (generic function with 1 method)

In [7]:
function combine_floatingbodies(floatingbodylist::Vector{FloatingBody},new_body_name::String)

    mesh_list = [floatingbody.mesh for floatingbody in floatingbodylist]
    dof_list = [floatingbody.dofs for floatingbody in floatingbodylist]  
    body_name_list = [replace(floatingbody.body_name, " " => "_") for floatingbody in floatingbodylist]
    num_face_list = [mesh.nfaces for mesh in mesh_list]
    cum_num_face_list = cumsum(num_face_list)
    tot_num_faces = sum(num_face_list)

    # New Mesh struct
    new_mesh = combine_meshes(mesh_list)

    # New FloatingBody dof_name and dof_value

    new_dof_keys = Symbol[]
    new_dof_mats = []
    for (body_index,body_name) in enumerate(body_name_list)
        # define nbf as cumalitive number of faces for previous body
        # This is used for shifting the location of the dof_mat 
        if body_index==1
            nbf = 0
        else
            nbf = cum_num_face_list[body_index-1] 
        end
        dofs = dof_list[body_index]
        for dof_name in keys(dofs)
            new_dof_key = Symbol(join([body_name,dof_name],"_"))
            dof_mat = dofs[dof_name]
            new_dof_mat = zeros(tot_num_faces,3)
            new_dof_mat[nbf+1:nbf+length(dof_mat[:,1]),:] = dof_mat  
            push!(new_dof_keys,new_dof_key)
            push!(new_dof_mats,new_dof_mat)        
        end
    end
    new_dofs = NamedTuple{tuple(new_dof_keys...)}(tuple(new_dof_mats...))

    return FloatingBody(new_mesh,new_dofs,new_body_name)
end

function combine_floatingbodies(floatingbodylist::Vector{FloatingBody})
    # New FloatingBody name
    body_name_list = [replace(floatingbody.body_name, " " => "_") for floatingbody in floatingbodylist]
    return combine_floatingbodies(floatingbodylist::Vector{FloatingBody},join(body_name_list,"_"))
end



combine_floatingbodies (generic function with 2 methods)

In [8]:
# Setup and solve BEM problems
cptarray = cptbody + cptbody2

# Setup and solve BEM problems
solver = cpt.BEMSolver()
dof_list = cptarray.dofs
xr = pyimport("xarray")
rad_dofs = string.(collect(keys(dof_list)))
test_matrix = xr.Dataset(coords=Dict("omega" => omegas, "wave_direction" => betas, "radiating_dof" => rad_dofs))
cptresults = cpt.BEMSolver().fill_dataset(test_matrix, cptarray, method="direct")



Solving BEM problems ---------------------------------------- 100% 0:00:00


PyObject <xarray.Dataset> Size: 2kB
Dimensions:              (omega: 2, radiating_dof: 4, influenced_dof: 4,
                          wave_direction: 2)
Coordinates:
    g                    float64 8B 9.81
    rho                  float64 8B 1e+03
    body_name            <U43 172B 'Horizontal Cylinder 1+Horizontal Cylinder 2'
    water_depth          float64 8B inf
    forward_speed        float64 8B 0.0
  * wave_direction       (wave_direction) float64 16B 0.0 0.5236
  * omega                (omega) float64 16B 1.03 1.7
  * radiating_dof        (radiating_dof) object 32B 'Horizontal Cylinder 1__H...
  * influenced_dof       (influenced_dof) object 32B 'Horizontal Cylinder 1__...
    period               (omega) float64 16B 6.1 3.696
    wavenumber           (omega) float64 16B 0.1081 0.2946
    wavelength           (omega) float64 16B 58.1 21.33
Data variables:
    added_mass           (omega, radiating_dof, influenced_dof) float64 256B ...
    radiation_damping    (omega, radiating_dof, influenced_dof) float64 256B ...
    diffraction_force    (omega, wave_direction, influenced_dof) complex128 256B ...
    Froude_Krylov_force  (omega, wave_direction, influenced_dof) complex128 256B ...
    excitation_force     (omega, wave_direction, influenced_dof) complex128 256B ...
Attributes: (12/16)
    start_of_computation:                     2026-03-20T13:21:48.258220
    green_function:                           Delhommeau
    tabulation_nr:                            676
    tabulation_rmax:                          100.0
    tabulation_nz:                            372
    tabulation_zmin:                          -251.0
    ...                                       ...
    gf_singularities:                         low_freq
    engine:                                   BasicMatrixEngine
    matrix_cache_size:                        1
    linear_solver:                            lu_decomposition
    creation_of_dataset:                      2026-03-20T13:21:48.278827
    capytaine_version:                        2.2.1

In [9]:
#Create and array (just another floating body)
fb_array = combine_floatingbodies(floatingbodylist)

# Create parameters NamedTuple
parameters = (wave_frequencies=omegas, 
wave_directions=betas,
radiating_dofs=collect(keys(fb_array.dofs)))

# Create DimStack of results
mhresults = compute_hydrodynamic_coefficients(parameters, fb_array)

┌ 4×4×2×2 DimStack ┐
├──────────────────┴───────────────────────────────────────────────────── dims ┐
  ↓ influenced_dofs Categorical{Symbol} [:Horizontal_Cylinder_1_Heave, …, :Horizontal_Cylinder_2_Pitch] ForwardOrdered,
  → radiating_dofs Categorical{Symbol} [:Horizontal_Cylinder_1_Heave, …, :Horizontal_Cylinder_2_Pitch] ForwardOrdered,
  ↗ wave_frequencies Sampled{Float64} [1.03, 1.7] ForwardOrdered Irregular Points,
  ⬔ wave_directions Sampled{Float64} [0.0, 0.5235987755982988] ForwardOrdered Irregular Points
├────────────────────────────────────────────────────────────────────── layers ┤
  :added_mass          eltype: Float64 dims: influenced_dofs, radiating_dofs, wave_frequencies size: 4×4×2
  :radiation_damping   eltype: Float64 dims: influenced_dofs, radiating_dofs, wave_frequencies size: 4×4×2
  :excitation_force    eltype: ComplexF64 dims: influenced_dofs, wave_frequencies, wave_directions size: 4×2×2
  :diffraction_force   eltype: ComplexF64 dims: influenced_dofs, wave_frequ

In [10]:
# Compare added_mass
omega_num = 2

display("Capytaine Added Mass")
display(cptresults.added_mass.sel(omega=omegas[omega_num]).values)

display("MarineHydro Added Mass")
display(collect(mhresults.added_mass[wave_frequencies = At(omegas[omega_num])]))

"Capytaine Added Mass"

4×4 Matrix{Float64}:
 5298.78   5303.08   -279.702  -328.441
 5302.84   8673.07   -233.719  -310.245
 -279.702  -230.962  5298.78   5294.48
 -325.684  -304.731  5294.72   8656.34

"MarineHydro Added Mass"

4×4 Matrix{Float64}:
 5306.26   5310.31   -279.22   -325.108
 5310.55   8678.83   -230.533  -304.198
 -279.22   -233.333  5306.26   5302.2
 -327.907  -309.798  5301.96   8662.14

In [18]:
rad_dofs[1]

:Horizontal_Cylinder_1_Heave

In [ ]:
#Create and array (just another floating body)
fb_array = combine_floatingbodies(floatingbodylist)

rad_dofs = collect(keys(fb_array.dofs))

# Create parameters NamedTuple
parameters = (wave_frequencies=[omegas[1]], 
wave_directions=[betas[1]],
radiating_dofs=[rad_dofs[1]])

# Create DimStack of results
mhresults = compute_hydrodynamic_coefficients(parameters, fb_array)

┌ 4×4×1×1 DimStack ┐
├──────────────────┴───────────────────────────────────────────────────── dims ┐
  ↓ influenced_dofs Categorical{Symbol} [:Horizontal_Cylinder_1_Heave, …, :Horizontal_Cylinder_2_Pitch] ForwardOrdered,
  → radiating_dofs Categorical{Symbol} [:Horizontal_Cylinder_1_Heave, …, :Horizontal_Cylinder_2_Pitch] ForwardOrdered,
  ↗ wave_frequencies Sampled{Float64} [1.03] ForwardOrdered Irregular Points,
  ⬔ wave_directions Sampled{Float64} [0.0] ForwardOrdered Irregular Points
├────────────────────────────────────────────────────────────────────── layers ┤
  :added_mass          eltype: Float64 dims: influenced_dofs, radiating_dofs, wave_frequencies size: 4×4×1
  :radiation_damping   eltype: Float64 dims: influenced_dofs, radiating_dofs, wave_frequencies size: 4×4×1
  :excitation_force    eltype: ComplexF64 dims: influenced_dofs, wave_frequencies, wave_directions size: 4×1×1
  :diffraction_force   eltype: ComplexF64 dims: influenced_dofs, wave_frequencies, wave_directions s

In [ ]:
display(collect(mhresults.added_mass[wave_frequencies = At(omegas[1])]))
# I should change code so influnced_dofs and radiating_dofs can be specified, but if not are specified, all A and B for dofs are computed
# also, inputs for not specifying wave dir should be more robust (only solve rad prob)

4×4 Matrix{Float64}:
 6824.2    0.0  0.0  0.0
 6824.59   0.0  0.0  0.0
 -240.981  0.0  0.0  0.0
 -270.536  0.0  0.0  0.0

In [ ]:
# Single dof differentiability

# Create FloatingBody struct
floatingbody1 = FloatingBody(mhmesh,
    [:Heave],
    [rotation_center...],
    "Horizontal Cylinder 1")

# Create parameters NamedTuple
parameters = (wave_frequencies=[omegas[1]], 
wave_directions=[betas[1]],
radiating_dofs=[:Heave])


function get_added_mass(omega)
    parameters = (wave_frequencies=[omega], 
    wave_directions=[betas[1]],
    radiating_dofs=[:Heave])
    mhresults = compute_hydrodynamic_coefficients(parameters, floatingbody1)
    added_mass = mhresults.added_mass[1]
    return added_mass
end


get_added_mass(1.5)

5760.545202167038

In [ ]:
# using Zygote
# A_w_grad, = Zygote.gradient(omega -> get_added_mass(omega),1.5)
# Must create seperate functions for assembling results based on dimarray or not
# Could also make seperate functions for solving just radiation or diffraction problems

ErrorException: Need an adjoint for constructor DimArray{Float64, 3, Tuple{Dim{:influenced_dofs, DimensionalData.Dimensions.Lookups.Categorical{Symbol, Vector{Symbol}, DimensionalData.Dimensions.Lookups.ForwardOrdered, DimensionalData.Dimensions.Lookups.NoMetadata}}, Dim{:radiating_dofs, DimensionalData.Dimensions.Lookups.Categorical{Symbol, Vector{Symbol}, DimensionalData.Dimensions.Lookups.ForwardOrdered, DimensionalData.Dimensions.Lookups.NoMetadata}}, Dim{:wave_frequencies, DimensionalData.Dimensions.Lookups.Sampled{Float64, Vector{Float64}, DimensionalData.Dimensions.Lookups.ForwardOrdered, DimensionalData.Dimensions.Lookups.Irregular{Tuple{Nothing, Nothing}}, DimensionalData.Dimensions.Lookups.Points, DimensionalData.Dimensions.Lookups.NoMetadata}}}, Tuple{}, Array{Float64, 3}, Symbol, DimensionalData.Dimensions.Lookups.NoMetadata}. Gradient is of type ChainRulesCore.InplaceableThunk{ChainRulesCore.Thunk{ChainRules.var"#675#677"{DimArray{Float64, 3, Tuple{Dim{:influenced_dofs, Di

In [ ]:
# to do: 
# Check differentiable
# Make differentiable wavebot

# Update all tests